# RACE Dataset EDA

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)


def resolve_raw_dir() -> Path:
    candidates = [Path('data/raw'), Path('/kaggle/working/data/raw'), Path('/kaggle/input')]
    target_files = ('train.csv', 'dev.csv', 'val.csv', 'test.csv')

    for candidate in candidates:
        if not candidate.exists():
            continue

        if candidate.is_dir() and any((candidate / filename).exists() for filename in target_files):
            return candidate

        if candidate.is_dir():
            for csv_file in candidate.rglob('train.csv'):
                parent = csv_file.parent
                if any((parent / filename).exists() for filename in target_files):
                    return parent

    return Path('data/raw')


RAW_DIR = resolve_raw_dir()
RAW_DIR

In [ ]:
split_files = {
    'train': 'train.csv',
    'dev': 'dev.csv',
    'val': 'val.csv',
    'test': 'test.csv',
}

def load_split(split_name: str) -> pd.DataFrame | None:
    filename = split_files.get(split_name, split_name)
    path = RAW_DIR / filename
    if path.exists():
        return pd.read_csv(path)
    return None

frames = {name: load_split(name) for name in ['train', 'dev', 'test']}
available = {name: frame for name, frame in frames.items() if frame is not None}
{name: frame.shape for name, frame in available.items()}

In [ ]:
train_df = available.get('train')
if train_df is not None:
    display(train_df.head())
    display(train_df.info())
else:
    print('train.csv was not found in the selected raw data directory.')

In [ ]:
if train_df is not None:
    missing = train_df.isna().sum().sort_values(ascending=False)
    display(missing[missing > 0])
else:
    print('Missing-value check skipped because the training split is unavailable.')

In [ ]:
def text_length(frame: pd.DataFrame, column: str) -> pd.Series:
    return frame[column].fillna('').astype(str).str.len()

if train_df is not None:
    summary = pd.DataFrame({
        'article_char_len': text_length(train_df, 'article'),
        'question_char_len': text_length(train_df, 'question'),
        'option_a_char_len': text_length(train_df, 'A'),
        'option_b_char_len': text_length(train_df, 'B'),
        'option_c_char_len': text_length(train_df, 'C'),
        'option_d_char_len': text_length(train_df, 'D'),
    })
    display(summary.describe().T)
else:
    print('Length summary skipped because the training split is unavailable.')

In [ ]:
if train_df is not None:
    answer_counts = train_df['answer'].value_counts().sort_index()
    ax = answer_counts.plot(kind='bar', figsize=(7, 4), color=['#3b82f6', '#10b981', '#f59e0b', '#ef4444'])
    ax.set_title('Answer Label Distribution')
    ax.set_xlabel('Correct Answer')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('Answer distribution plot skipped because the training split is unavailable.')

In [ ]:
if train_df is not None:
    def word_count(value: object) -> int:
        text = re.sub(r'\s+', ' ', str(value).lower()).strip()
        return 0 if not text else len(text.split(' '))

    article_words = train_df['article'].fillna('').map(word_count)
    question_words = train_df['question'].fillna('').map(word_count)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(article_words, bins=40, ax=axes[0], color='#2563eb')
    axes[0].set_title('Article Word Count')
    sns.histplot(question_words, bins=40, ax=axes[1], color='#16a34a')
    axes[1].set_title('Question Word Count')
    plt.tight_layout()
    plt.show()
else:
    print('Length histograms skipped because the training split is unavailable.')